# The Gradient & Gradient Descent
**Topic:** Calculus for Machine Learning

In [1]:
import numpy as np
import pandas as pd

import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

import ipywidgets as widgets
from ipywidgets import interact, interactive, fixed, interact_manual
from ipywidgets import IntSlider, FloatSlider, Dropdown, Button, Output, HBox, VBox, Label

from IPython.display import display, HTML, clear_output
from scipy import stats
from tkh_utils import PALETTE, FONT, base_layout


---
## What you'll explore

By the end of this demo you will be able to:

- **Describe** gradient descent as stepping downhill on a loss surface by following the negative gradient
- **Explain** the role of the learning rate and what happens when it is too large or too small
- **Interpret** a loss curve over training steps and identify signs of divergence, slow convergence, and successful training

> **Tip:** Start by moving the **learning rate slider** to a very large value and clicking Play — watch the ball overshoot the minimum and possibly diverge. Then try a very small value and watch it creep toward the bottom. The sweet spot is in between.

---
## How we got here

In *05: Partial Derivatives* we learned that the gradient is a vector made of all partial derivatives, pointing in the direction of steepest increase. In *03: Derivatives* we saw that the minimum of a function is where all derivatives are zero. Gradient descent connects these ideas: start anywhere, measure the gradient (steepness and direction), take a small step in the opposite direction (downhill), and repeat until the gradient is (close to) zero.

---
## Why this matters for data science

Gradient descent is the optimization algorithm that trains linear regression, logistic regression, support vector machines, and every neural network. When you call `model.fit(X, y)` in scikit-learn or `model.train()` in PyTorch, gradient descent (or one of its variants) is running underneath. The learning rate is the single most important hyperparameter to tune in most deep learning models. Understanding what it controls makes you a much more effective practitioner.

---
## Try it yourself

In [2]:
np.random.seed(42)

def loss_fn(w):
    return (w - 3.0) ** 2 + 2.0

def grad_fn(w):
    return 2.0 * (w - 3.0)

W_INIT  = -1.5
w_range = np.linspace(-2.5, 8.5, 300)

out_curve   = Output()
out_history = Output()

lr_slider = FloatSlider(value=0.3, min=0.01, max=1.9, step=0.05,
    description="Learning rate:",
    style={"description_width": "120px"},
    layout=widgets.Layout(width="420px"))

step_btn  = Button(description="▶  Step",  button_style="primary",
    layout=widgets.Layout(width="110px"))
reset_btn = Button(description="↺  Reset", button_style="",
    layout=widgets.Layout(width="110px"))
step_label = Label(value=f"Step 0  |  Loss: {loss_fn(W_INIT):.4f}")

state = {"w": W_INIT, "history": [(W_INIT, loss_fn(W_INIT))]}

def render():
    w_cur  = state["w"]
    hist   = state["history"]
    g      = grad_fn(w_cur)
    n_step = len(hist) - 1
    xs = [h[0] for h in hist]
    ys = [h[1] for h in hist]

    traces1 = [
        go.Scatter(x=w_range, y=loss_fn(w_range), mode="lines",
            line=dict(color=PALETTE["muted"], width=2.5), name="Loss f(w)"),
        go.Scatter(x=xs, y=ys, mode="lines+markers",
            line=dict(color=PALETTE["primary"], width=1.5, dash="dot"),
            marker=dict(size=6, color=PALETTE["primary"], opacity=0.5),
            name="Path taken"),
        go.Scatter(x=[w_cur], y=[loss_fn(w_cur)], mode="markers",
            marker=dict(size=14, color=PALETTE["secondary"], symbol="circle"),
            name=f"w = {w_cur:.3f}"),
    ]
    layout1 = base_layout(
        title=f"Step {n_step}  |  w = {w_cur:.3f}  |  Loss = {loss_fn(w_cur):.4f}  |  Gradient = {g:.3f}",
        xaxis_title="Parameter w", yaxis_title="Loss")
    with out_curve:
        clear_output(wait=True)
        display(go.FigureWidget(go.Figure(data=traces1, layout=layout1)))

    traces2 = [
        go.Scatter(x=list(range(len(ys))), y=ys, mode="lines+markers",
            line=dict(color=PALETTE["accent"], width=2),
            marker=dict(size=5), name="Loss history"),
    ]
    layout2 = base_layout(
        title="Loss over steps", xaxis_title="Step", yaxis_title="Loss")
    with out_history:
        clear_output(wait=True)
        display(go.FigureWidget(go.Figure(data=traces2, layout=layout2)))

def on_step(btn):
    w     = state["w"]
    w_new = w - lr_slider.value * grad_fn(w)
    state["w"] = w_new
    state["history"].append((w_new, loss_fn(w_new)))
    n = len(state["history"]) - 1
    step_label.value = f"Step {n}  |  Loss: {loss_fn(w_new):.4f}"
    render()

def on_reset(btn):
    state["w"] = W_INIT
    state["history"] = [(W_INIT, loss_fn(W_INIT))]
    step_label.value = f"Step 0  |  Loss: {loss_fn(W_INIT):.4f}"
    render()

step_btn.on_click(on_step)
reset_btn.on_click(on_reset)

display(VBox([lr_slider, HBox([step_btn, reset_btn, step_label]), out_curve, out_history]))
render()

---
## What's happening?

Gradient descent is an iterative algorithm. At each step, it measures the gradient of the loss with respect to all parameters and moves each parameter in the direction that reduces the loss.

The update rule for a single weight w is:

$$w_{\text{new}} = w_{\text{old}} - \alpha \cdot \frac{\partial \text{Loss}}{\partial w}$$

Here α (alpha) is the **learning rate**: a small positive number that controls the step size. The minus sign is the crucial part, we subtract the gradient, because the gradient points uphill and we want to go downhill.

| Learning rate | Behavior | Risk |
|--------------|---------|------|
| Too small (< 0.001) | Tiny steps; convergence is correct but very slow | Wastes compute; may not converge in budget |
| Good (0.001 to 0.1) | Smooth descent; reaches minimum in reasonable steps | None: this is the target range |
| Too large (> 1.0) | Overshoots the minimum; bounces or diverges | Loss increases; training fails completely |
| Adaptive (Adam, RMSprop) | Learning rate adjusts automatically per parameter | More complex to understand; usually best in practice |

### What convergence looks like

A well-trained model shows a loss curve that decreases steeply at first, then flattens as it approaches the minimum. Once the gradient is nearly zero, the updates become tiny and the loss stops meaningfully decreasing. That is when training is complete.

Move the learning rate slider to find the value that reaches the minimum fastest without overshooting.

---
## Real-world example: Training loss curves for a neural network

The chart shows three training runs of the same neural network with different learning rates. This is one of the most common diagnostic charts in deep learning: looking at the loss curve to diagnose what went wrong with a training run.

Notice:

- **Notice:** The 0.001 learning rate curve decreases smoothly but slowly; after 100 steps it still has significant loss remaining because the steps are too small to have gotten far
- **Notice:** The 0.1 learning rate curve drops quickly at first then plateaus at a good minimum; this is the successful run
- **Notice:** The 5.0 learning rate curve initially drops but then shoots upward as the large steps overshoot the minimum and send the weights to a worse region; training has diverged

> **Discussion question:** A teammate reports that their model's training loss decreased for 50 epochs then started increasing again. They call this "divergence." Is that the only possible explanation? What else could cause loss to increase after initially decreasing?

In [3]:
np.random.seed(42)

def simulate_gd(lr, n_steps=100, w_init=4.0):
    # Gradient descent on f(w) = w^2 (minimum at w=0).
    w = w_init
    losses = []
    for _ in range(n_steps):
        grad  = 2 * w      # derivative of w^2
        w     = w - lr * grad
        losses.append(w ** 2)
    return losses

steps  = np.arange(1, 101)
configs = [(0.001, PALETTE["muted"],      "lr=0.001 (too slow)"),
           (0.10,  PALETTE["accent"],     "lr=0.1   (just right)"),
           (5.0,   PALETTE["secondary"],  "lr=5.0   (diverges)")]

fig = go.Figure()
for lr, color, label in configs:
    losses = simulate_gd(lr)
    fig.add_trace(go.Scatter(x=steps, y=losses, mode="lines",
                             line=dict(color=color, width=2.5), name=label))
layout = base_layout(
    title="Gradient Descent Loss Curves — Three Learning Rates on the Same Problem",
    xaxis_title="Training Step", yaxis_title="Loss",
)
layout.update(yaxis=dict(range=[-0.5, 18]))
fig.update_layout(layout)
fig.show()

### Learning rate and optimizer comparison

| Learning rate | Behavior | Risk | When to use it |
|--------------|---------|------|---------------|
| Fixed small (0.001) | Slow, stable descent | Very slow convergence | Early experiments, small models |
| Fixed medium (0.01–0.1) | Fast, stable for simple problems | May overshoot for complex loss surfaces | Convex problems, linear models |
| Fixed large (> 1.0) | Fast initial drop, then divergence | Training fails | Almost never: reduce immediately |
| Learning rate schedule | Starts large, decreases over time | Requires tuning the schedule | Deep learning standard practice |
| Adam optimizer | Adaptive per-parameter rate | Slightly more memory | Default for most neural networks |
| SGD with momentum | Builds up speed in consistent directions | Overshoots sharp curves | When you need fine control |

---
## Key takeaway

> **Gradient descent repeatedly steps in the direction opposite to the gradient, shrunk by the learning rate; too large a step overshoots the minimum, too small a step wastes compute, and the right rate reaches it efficiently.**

---
*Next up: Integrals & Area Under the Curve — the other half of calculus, and why it shows up in probability, ROC curves, and model evaluation*